# Data preprocessing

Data is gathered from Udemy course on credit risk.

## Importing packages

In [1]:
import pandas as pd
import numpy as np

## Loading data

In [2]:
df = pd.read_csv(r"C:\Users\niels\Lending Club project\Lending-Club-credit-risk-modelling\data\loan_data_2007_2014.csv", keep_default_na=False, na_values=[''])

C:\Users\niels\AppData\Local\Temp\ipykernel_16776\1250826200.py:1: DtypeWarning: Columns (0: desc) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\niels\Lending Club project\Lending-Club-credit-risk-modelling\data\loan_data_2007_2014.csv", keep_default_na=False, na_values=[''])


## Creating default definition

Unfortenately it seems that each observation is captured when the observations are already in default meaning any attempt of PD-modelling is more a default detection system than an actually probability of default model. Nevertheless we will calibrate the model to the average defaultrate of U.S. banks and realisticially it will still capture behaviour that would also be present for customer whom defaults within 12 months.

The columns of loan status is as follows:

In [3]:
print(str(list(pd.unique(df["loan_status"]))).strip("[]"))

'Fully Paid', 'Charged Off', 'Current', 'Default', 'Late (31-120 days)', 'In Grace Period', 'Late (16-30 days)', 'Does not meet the credit policy. Status:Fully Paid', 'Does not meet the credit policy. Status:Charged Off'


Note that there is no missings in this column.

In our definition of default we consider the loan default if the status is:

* Charged Off
* Default
* Does not meet the credit policy. Status:Charged Off

In [4]:
default = ["Charged Off", "Default", 'Does not meet the credit policy. Status:Charged Off']

Now we can define the target (default being 1 and non-default being 0).

In [5]:
df["default"] = [1 if s in default else 0 for s in df["loan_status"]]

The default of the data can be calculated as follows:

In [6]:
print("Defaultrate across the data: " + str(round(100*sum(df["default"])/len(df["default"]),2)) + "%")

Defaultrate across the data: 9.45%


The defaultrate is unusually large and this is a symptom of us modelling primo defaults rather than modelling defaults the coming 12 months.

## Removing redundant columns

Some of the columns in the data only have missing values hence are redundant.

In [7]:
df = df[[col for col in df.columns if sum(pd.isna(df[col])) != df.shape[0]]].copy()

In [8]:
len(pd.unique(df["member_id"])) == df.shape[0]

True

Since each row correspond to a different member id the index column and id are also redundant.

In [9]:
df.drop(columns=["Unnamed: 0", "id"], inplace = True)

In [10]:
d = {}

for col in df.columns:
    d[col] = len(pd.unique(df[col]))

uniques_count = pd.Series(d)
uniques_count.name = 'Unqiue count'
display(uniques_count)

member_id                      466285
loan_amnt                        1352
funded_amnt                      1354
funded_amnt_inv                  9854
term                                2
int_rate                          506
installment                     55622
grade                               7
sub_grade                          35
emp_title                      205476
emp_length                         12
home_ownership                      6
annual_inc                      31902
verification_status                 3
issue_d                            91
loan_status                         9
pymnt_plan                          2
url                            466285
desc                           124437
purpose                            14
title                           63100
zip_code                          888
addr_state                         50
dti                              3997
delinq_2yrs                        25
earliest_cr_line                  665
inq_last_6mt

In [11]:
drops = [col for col in uniques_count.index if uniques_count[col].item() == 1]

In [12]:
df.drop(columns= drops + ["url", "desc" ], inplace = True)

## Data types

In [13]:
df.dtypes

member_id                        int64
loan_amnt                        int64
funded_amnt                      int64
funded_amnt_inv                float64
term                               str
int_rate                       float64
installment                    float64
grade                              str
sub_grade                          str
emp_title                          str
emp_length                         str
home_ownership                     str
annual_inc                     float64
verification_status                str
issue_d                            str
loan_status                        str
pymnt_plan                         str
purpose                            str
title                              str
zip_code                           str
addr_state                         str
dti                            float64
delinq_2yrs                    float64
earliest_cr_line                   str
inq_last_6mths                 float64
mths_since_last_delinq   

There is no object data type which is usually an indication of mixed elements, however it is worth noting that the date columns such as issue_d are of string type and on the form:

In [14]:
df["issue_d"][0]

'Dec-11'

i.e. month and then last digits of the years. This is an issue because years before the 2000s will likely be greater than after the 2000s so any order is destroyed in the pure column. This ordering will be fixed.

In [15]:
date_columns = ["last_credit_pull_d", "last_pymnt_d", "next_pymnt_d", "issue_d", "earliest_cr_line"]

In [16]:
for date in date_columns:
    print(sorted(list(pd.unique(df[date].str.slice(4).apply(lambda s: float(s))))))

[np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(nan)]
[np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(nan)]
[np.float64(nan), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0)]
[np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0)]
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(44.0), np.float64(46.0), np.float64(48.0), np.float64(4

Checking the lists yields that earliest year in 1900s is 1944

In [17]:
for date in date_columns:
    df[date] = np.where(df[date].str.slice(start=4) <= '44',
                        '01-' + df[date].str.slice(stop=4) + '20' + df[date].str.slice(start=4),
                        '01-' + df[date].str.slice(stop=4) + '19' + df[date].str.slice(start=4)
                        )

In [18]:
month_map = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
}

for date in date_columns:
    s = df[date].astype(str).str.strip()

    for mon, num in month_map.items():
        s = s.str.replace(mon, num, regex=False)

    df[date] = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")

## Splitting the data into a training and test data

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
train, test = train_test_split(df, test_size = 0.8, random_state = 42, shuffle = True)

In [21]:
train.to_csv(r"C:\Users\niels\Lending Club project\Lending-Club-credit-risk-modelling\data\train.csv", index=False)
test.to_csv(r"C:\Users\niels\Lending Club project\Lending-Club-credit-risk-modelling\data\test.csv", index=False)